In [2]:
!pip install sentence-transformers faiss-cpu pandas numpy

In [3]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

def find_ai_lab_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current] + list(current.parents):
        if candidate.name == "ai_lab":
            return candidate
    raise RuntimeError("Không tìm thấy thư mục ai_lab.")

AI_LAB_ROOT = find_ai_lab_root(Path.cwd())

ARTIFACTS_DIR = AI_LAB_ROOT / "artifacts" / "retriever_v1"
REPORTS_DIR = AI_LAB_ROOT / "reports"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

KB_CHUNKS_PATH = ARTIFACTS_DIR / "kb_chunks_v1.json"
CHUNK_METADATA_PATH = ARTIFACTS_DIR / "chunk_metadata.json"

EMBEDDINGS_PATH = ARTIFACTS_DIR / "chunk_embeddings.npy"
FAISS_INDEX_PATH = ARTIFACTS_DIR / "faiss.index"
EMBEDDING_CONFIG_PATH = ARTIFACTS_DIR / "embedding_config.json"
RETRIEVER_MANIFEST_PATH = ARTIFACTS_DIR / "retriever_manifest.json"

print("AI_LAB_ROOT =", AI_LAB_ROOT)
print("KB_CHUNKS_PATH =", KB_CHUNKS_PATH)

AI_LAB_ROOT = D:\homelab\ai_lab
KB_CHUNKS_PATH = D:\homelab\ai_lab\artifacts\retriever_v1\kb_chunks_v1.json


In [4]:
with open(KB_CHUNKS_PATH, "r", encoding="utf-8") as f:
    kb_chunks = json.load(f)

with open(CHUNK_METADATA_PATH, "r", encoding="utf-8") as f:
    chunk_metadata = json.load(f)

print("Số chunks:", len(kb_chunks))
pd.DataFrame(kb_chunks)[["chunk_id", "title", "section", "risk_level"]]

Số chunks: 15


,chunk_id,title,section,risk_level
0,kb_v1_001_c1,Khi có dấu hiệu nhiễm trùng và cảm giác rất mệ...,red_flags,high
1,kb_v1_002_c1,Một số dấu hiệu bên ngoài làm tăng mức độ cảnh...,red_flags,high
2,kb_v1_003_c1,Một số dấu hiệu có thể làm mức độ lo ngại tăng...,red_flags,high
3,kb_v1_004_c1,Khi có dấu hiệu nguy cơ cao của nhiễm trùng nặ...,red_flags,high
4,kb_v1_005_c1,Xét nghiệm máu là gì,test_explainers,low
5,kb_v1_006_c1,Vì sao bác sĩ có thể chỉ định xét nghiệm máu,test_explainers,low
6,kb_v1_007_c1,Một số loại xét nghiệm máu thường gặp,test_explainers,low
7,kb_v1_008_c1,Cần chuẩn bị gì trước khi xét nghiệm máu,test_explainers,low
8,kb_v1_009_c1,Sau khi xét nghiệm máu và khi nhận kết quả,test_explainers,low
9,kb_v1_010_c1,Khi nào đau ngực cần gọi cấp cứu ngay,red_flags,high


In [5]:
def build_passage_text(chunk):
    return "passage: " + chunk["chunk_text"].strip()

chunk_ids = [c["chunk_id"] for c in kb_chunks]
texts = [build_passage_text(c) for c in kb_chunks]

print("Ví dụ chunk text đầu tiên:\n")
print(texts[0][:1200])

Ví dụ chunk text đầu tiên:

passage: Tiêu đề: Khi có dấu hiệu nhiễm trùng và cảm giác rất mệt hoặc rất không ổn, cần được đánh giá sớm
Mục: red_flags
Mức độ rủi ro: high
Từ khóa: nhiễm trùng nặng, sepsis, dấu hiệu nặng
Nhãn: sepsis, red_flags, infection
Loại FAQ: red_flag_general
Lưu ý an toàn: Không dùng nội dung này để tự kết luận sepsis.
Nội dung: Nếu một người có dấu hiệu nhiễm trùng và đồng thời cảm thấy rất mệt, rất yếu hoặc diễn tiến xấu nhanh, cần nghĩ đến khả năng bệnh nặng hơn bình thường và nên được đánh giá y tế sớm.


In [6]:
MODEL_NAME = "intfloat/multilingual-e5-small"
model = SentenceTransformer(MODEL_NAME)
print("Loaded model:", MODEL_NAME)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Loaded model: intfloat/multilingual-e5-small


In [7]:
embeddings = model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings shape:", embeddings.shape)
print("Dtype:", embeddings.dtype)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (15, 384)
Dtype: float32


In [8]:
np.save(EMBEDDINGS_PATH, embeddings)
print("Đã ghi:", EMBEDDINGS_PATH)

Đã ghi: D:\homelab\ai_lab\artifacts\retriever_v1\chunk_embeddings.npy


In [9]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype("float32"))

faiss.write_index(index, str(FAISS_INDEX_PATH))
print("Đã ghi:", FAISS_INDEX_PATH)
print("Tổng số vector trong index:", index.ntotal)

Đã ghi: D:\homelab\ai_lab\artifacts\retriever_v1\faiss.index
Tổng số vector trong index: 15


In [10]:
embedding_config = {
    "model_name": MODEL_NAME,
    "embedding_dimension": int(embeddings.shape[1]),
    "normalized": True,
    "index_type": "IndexFlatIP",
    "text_field": "chunk_text",
    "passage_prefix": "passage: ",
    "query_prefix": "query: "
}

with open(EMBEDDING_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(embedding_config, f, ensure_ascii=False, indent=2)

print("Đã ghi:", EMBEDDING_CONFIG_PATH)

Đã ghi: D:\homelab\ai_lab\artifacts\retriever_v1\embedding_config.json


In [11]:
retriever_manifest = {
    "retriever_version": "v1",
    "kb_file": "kb_chunks_v1.json",
    "metadata_file": "chunk_metadata.json",
    "embeddings_file": "chunk_embeddings.npy",
    "faiss_index_file": "faiss.index",
    "embedding_config_file": "embedding_config.json",
    "chunk_count": len(kb_chunks),
    "model_name": MODEL_NAME,
    "top_k_default": 3
}

with open(RETRIEVER_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(retriever_manifest, f, ensure_ascii=False, indent=2)

print("Đã ghi:", RETRIEVER_MANIFEST_PATH)

Đã ghi: D:\homelab\ai_lab\artifacts\retriever_v1\retriever_manifest.json


In [12]:
def search(query, top_k=3):
    query_text = "query: " + query.strip()
    q_emb = model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(q_emb, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        chunk = kb_chunks[idx]
        results.append({
            "rank": len(results) + 1,
            "score": float(score),
            "chunk_id": chunk["chunk_id"],
            "title": chunk["title"],
            "section": chunk["section"],
            "source_id": chunk["source_id"],
            "content_preview": chunk["content"][:300]
        })
    return results

test_queries = [
    "xét nghiệm máu là gì",
    "khi nào đau ngực cần đi cấp cứu",
    "khó thở khi nào nguy hiểm",
    "dấu hiệu nhiễm trùng nặng cần đi viện"
]

for q in test_queries:
    print("=" * 100)
    print("QUERY:", q)
    results = search(q, top_k=3)
    for r in results:
        print(f"[{r['rank']}] score={r['score']:.4f} | {r['title']} | {r['section']} | {r['source_id']}")
    print()

QUERY: xét nghiệm máu là gì
[1] score=0.9397 | Xét nghiệm máu là gì | test_explainers | blood_tests
[2] score=0.9115 | Vì sao bác sĩ có thể chỉ định xét nghiệm máu | test_explainers | blood_tests
[3] score=0.9038 | Một số loại xét nghiệm máu thường gặp | test_explainers | blood_tests

QUERY: khi nào đau ngực cần đi cấp cứu
[1] score=0.9324 | Khi nào đau ngực cần gọi cấp cứu ngay | red_flags | chest_pain
[2] score=0.8977 | Khi nào đau ngực nên đi khám sớm | red_flags | chest_pain
[3] score=0.8808 | Khi nào khó thở cần hỗ trợ khẩn cấp | red_flags | shortness_of_breath

QUERY: khó thở khi nào nguy hiểm
[1] score=0.9085 | Khi nào khó thở cần hỗ trợ khẩn cấp | red_flags | shortness_of_breath
[2] score=0.9024 | Khi nào khó thở cần được đánh giá y tế sớm | red_flags | shortness_of_breath
[3] score=0.8864 | Không nên tự chẩn đoán nguyên nhân khó thở | red_flags | shortness_of_breath

QUERY: dấu hiệu nhiễm trùng nặng cần đi viện
[1] score=0.9100 | Khi có dấu hiệu nhiễm trùng và cảm giác rất mệt

In [13]:
rows = []
for q in test_queries:
    results = search(q, top_k=3)
    for r in results:
        rows.append({
            "query": q,
            "rank": r["rank"],
            "score": r["score"],
            "chunk_id": r["chunk_id"],
            "title": r["title"],
            "section": r["section"],
            "source_id": r["source_id"]
        })

df_smoke = pd.DataFrame(rows)
df_smoke

,query,rank,score,chunk_id,title,section,source_id
0,xét nghiệm máu là gì,1,0.939669,kb_v1_005_c1,Xét nghiệm máu là gì,test_explainers,blood_tests
1,xét nghiệm máu là gì,2,0.911544,kb_v1_006_c1,Vì sao bác sĩ có thể chỉ định xét nghiệm máu,test_explainers,blood_tests
2,xét nghiệm máu là gì,3,0.903754,kb_v1_007_c1,Một số loại xét nghiệm máu thường gặp,test_explainers,blood_tests
3,khi nào đau ngực cần đi cấp cứu,1,0.932418,kb_v1_010_c1,Khi nào đau ngực cần gọi cấp cứu ngay,red_flags,chest_pain
4,khi nào đau ngực cần đi cấp cứu,2,0.897734,kb_v1_011_c1,Khi nào đau ngực nên đi khám sớm,red_flags,chest_pain
5,khi nào đau ngực cần đi cấp cứu,3,0.880756,kb_v1_013_c1,Khi nào khó thở cần hỗ trợ khẩn cấp,red_flags,shortness_of_breath
6,khó thở khi nào nguy hiểm,1,0.908482,kb_v1_013_c1,Khi nào khó thở cần hỗ trợ khẩn cấp,red_flags,shortness_of_breath
7,khó thở khi nào nguy hiểm,2,0.902402,kb_v1_014_c1,Khi nào khó thở cần được đánh giá y tế sớm,red_flags,shortness_of_breath
8,khó thở khi nào nguy hiểm,3,0.886411,kb_v1_015_c1,Không nên tự chẩn đoán nguyên nhân khó thở,red_flags,shortness_of_breath
9,dấu hiệu nhiễm trùng nặng cần đi viện,1,0.910017,kb_v1_001_c1,Khi có dấu hiệu nhiễm trùng và cảm giác rất mệ...,red_flags,nice_sepsis_overview
